In [36]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [37]:
depmap = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/DepMap/gene_essentiality_data.h5ad")

In [38]:
# map gene symbols to ensemble ids
gene_info = pd.read_csv("/cluster/work/boeva/eheiss/scbFM/data/bulkformer_gene_info.csv")
sym2ensg = gene_info.set_index("gene_symbol")["ensg_id"].to_dict()

original_symbols = depmap.var_names.tolist()
mapped_ensg = [sym2ensg.get(sym, None) for sym in original_symbols]

n_mapped = sum(e is not None for e in mapped_ensg)
n_unmapped = sum(e is None for e in mapped_ensg)
print(f"Mapped: {n_mapped} / {len(original_symbols)}  |  Unmapped: {n_unmapped}")

# Keep only genes that could be mapped
mask = np.array([e is not None for e in mapped_ensg])
depmap = depmap[:, mask].copy()
depmap.var_names = [e for e in mapped_ensg if e is not None]

Mapped: 17742 / 17862  |  Unmapped: 120


In [39]:
depmap.write("/cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad")

## Part II - Statistics

In [41]:
with open("/cluster/work/boeva/eheiss/scbFM/data/gene_list.txt") as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [42]:
depmap = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad")
print(depmap)

AnnData object with n_obs × n_vars = 1103 × 17742
    obs: 'cell_line_display_name', 'lineage_1', 'lineage_2', 'lineage_3', 'lineage_6', 'lineage_4'
    layers: 'essen_array', 'expr_array'


In [43]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(depmap.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, depmap.n_obs, chunk_size):
    end = min(start + chunk_size, depmap.n_obs)

    X_chunk = depmap.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == depmap.n_obs:
        print(f"Processed {n_obs_done:,}/{depmap.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


Processed 1,000/1,103 samples
Processed 1,103/1,103 samples
Average portion of non-zero genes NOT in gene_list: 0.22118710156980184
Average portion of total reads NOT in gene_list: 0.1304624871424092


## Part III - filter to gene list

In [46]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(depmap.var_names.astype(str))

reorder_idx = gene_index.get_indexer(gene_list)
missing = [g for g, i in zip(gene_list, reorder_idx) if i < 0]
if missing:
    raise ValueError(f"{len(missing)} genes from gene_list are missing in depmap. First 20: {missing[:20]}")

out_path = "/cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad"
chunk_size = 1000

chunks = []

for start in range(0, depmap.n_obs, chunk_size):
    end = min(start + chunk_size, depmap.n_obs)
    X_chunk = depmap.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC handles column selection more reliably than CSR on some SciPy builds.
        X_chunk = X_chunk.tocsc()[:, reorder_idx].tocsr()
    else:
        X_chunk = np.asarray(X_chunk)[:, reorder_idx]

    chunk = ad.AnnData(
        X=X_chunk,
        obs=depmap.obs.iloc[start:end].copy(),
        var=pd.DataFrame(index=pd.Index(gene_list, name=depmap.var_names.name)),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{depmap.n_obs:,} samples")

depmap_aligned = ad.concat(chunks, axis=0, join="inner", merge="same")
depmap_aligned.var_names = pd.Index(gene_list, name=depmap.var_names.name)

depmap_aligned.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", depmap_aligned.shape)

Prepared 1,000/1,103 samples
Prepared 1,103/1,103 samples
Wrote: /cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad
Shape: (1103, 13004)
